# SAMSum Dataset — Exploratory Data Analysis

**Project:** Meeting Summarizer · Dialogue Abstractive Summarization  
**Dataset:** SAMSum (`knkarthick/samsum`) — **CC BY-NC-ND 4.0 (non-commercial only)**  
**Hardware:** Apple M4 Pro · 24 GB UMA · MPS / BF16  
**Python:** 3.12.x venv `~/.venvs/meeting-summarizer`

---

This notebook is the Day 1 analytical companion to `scripts/data_audit.py`.  
It provides interactive visualizations and deeper inspection of the same statistics  
that `data_audit.py` writes to `results/metrics/data_audit.json`.

**Sections:**
1. Load SAMSum from local cache (no network)
2. Dataset structure inspection
3. Token-length distribution (T5 tokenizer)
4. Speaker count distribution
5. Example dialogues and summaries
6. Compression ratio analysis
7. Length correlation heatmap
8. Key findings summary

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json
import os
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Enforce offline mode — all data must be pre-cached
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 4)

PROJECT_ROOT = Path.cwd().parent   # notebooks/ → project root
print(f"Project root : {PROJECT_ROOT}")
print(f"Offline mode : HF_DATASETS_OFFLINE={os.environ['HF_DATASETS_OFFLINE']}")

## 1. Load SAMSum from Local Cache

In [ ]:
from datasets import load_dataset

ds = load_dataset("knkarthick/samsum")   # reads from ~/.cache/huggingface/datasets/

print(ds)
print(f"\nSplits: {list(ds.keys())}")
print(f"  train      : {len(ds['train']):,}")
print(f"  validation : {len(ds['validation']):,}")
print(f"  test       : {len(ds['test']):,}")
print(f"\nColumns: {ds['train'].column_names}")
print(f"Features: {ds['train'].features}")

## 2. Dataset Structure — Example Inspection

In [ ]:
# ── Data leakage guard ────────────────────────────────────────────────────────
train_ids = set(ds["train"]["id"])
val_ids   = set(ds["validation"]["id"])
test_ids  = set(ds["test"]["id"])

assert len(train_ids & val_ids)  == 0, "DATA LEAKAGE: train ∩ val"
assert len(train_ids & test_ids) == 0, "DATA LEAKAGE: train ∩ test"
assert len(val_ids   & test_ids) == 0, "DATA LEAKAGE: val ∩ test"
print("✅ Zero ID overlap across all splits — no data leakage")

# ── Inspect one example ──────────────────────────────────────────────────────
ex = ds["train"][42]
print(f"\n{'─'*60}")
print(f"ID      : {ex['id']}")
print(f"\nDIALOGUE:\n{ex['dialogue']}")
print(f"\nSUMMARY:\n{ex['summary']}")
print(f"{'─'*60}")

## 3. Token-Length Distribution (T5 Tokenizer)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

# Tokenize train split
print("Tokenizing train split...", end=" ", flush=True)
dial_lens = [len(tokenizer(d, truncation=False)["input_ids"]) for d in ds["train"]["dialogue"]]
summ_lens = [len(tokenizer(s, truncation=False)["input_ids"]) for s in ds["train"]["summary"]]
print(f"done ({len(dial_lens):,} examples)")

def pct_stats(arr):
    a = np.array(arr)
    return {k: int(v) for k, v in zip(
        ["min", "p50", "p90", "p95", "p99", "max"],
        [a.min(), np.percentile(a,50), np.percentile(a,90),
         np.percentile(a,95), np.percentile(a,99), a.max()]
    )} | {"mean": round(float(a.mean()),1), "std": round(float(a.std()),1)}

dial_stats = pct_stats(dial_lens)
summ_stats = pct_stats(summ_lens)

df_stats = pd.DataFrame([dial_stats, summ_stats], index=["Dialogue tokens", "Summary tokens"])
display(df_stats)

print(f"\n→ max_source_length=512 covers: {sum(1 for l in dial_lens if l <= 512)/len(dial_lens):.2%} of train dialogues")
print(f"→ max_target_length=128 covers: {sum(1 for l in summ_lens if l <= 128)/len(summ_lens):.2%} of train summaries")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(dial_lens, bins=60, color="#4878CF", edgecolor="white", linewidth=0.4)
axes[0].axvline(512, color="red",    linestyle="--", linewidth=1.5, label="max_source_length=512")
axes[0].axvline(dial_stats["p99"],  color="orange", linestyle="--", linewidth=1.5,
                label=f"p99={dial_stats['p99']}")
axes[0].set_title("Dialogue Token Lengths (train)", fontsize=12)
axes[0].set_xlabel("Tokens"); axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(summ_lens, bins=40, color="#6ACC65", edgecolor="white", linewidth=0.4)
axes[1].axvline(128, color="red",    linestyle="--", linewidth=1.5, label="max_target_length=128")
axes[1].axvline(summ_stats["max"],  color="orange", linestyle="--", linewidth=1.5,
                label=f"max={summ_stats['max']}")
axes[1].set_title("Summary Token Lengths (train)", fontsize=12)
axes[1].set_xlabel("Tokens"); axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "metrics" / "token_length_distributions.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/metrics/token_length_distributions.png")

## 4. Speaker Count Distribution

In [ ]:
def count_speakers(dialogue: str) -> int:
    speakers = {re.match(r"^([^\n:]+):", line).group(1).strip()
                for line in dialogue.strip().split("\n")
                if re.match(r"^([^\n:]+):", line)}
    return max(len(speakers), 1)

speaker_counts = [count_speakers(d) for d in ds["train"]["dialogue"]]
count_dist     = Counter(speaker_counts)
total          = len(speaker_counts)

df_spk = pd.DataFrame([
    {"Speakers": k, "Count": v, "Percentage": round(100*v/total, 1)}
    for k, v in sorted(count_dist.items())
])
display(df_spk)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar([str(k) for k in sorted(count_dist)],
       [count_dist[k] for k in sorted(count_dist)],
       color=sns.color_palette("muted", len(count_dist)))
ax.set_xlabel("Number of Speakers")
ax.set_ylabel("Dialogues")
ax.set_title("Speaker Count Distribution (train split)")
for i, k in enumerate(sorted(count_dist)):
    ax.text(i, count_dist[k] + 30, f"{100*count_dist[k]/total:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "metrics" / "speaker_distribution.png",
            dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/metrics/speaker_distribution.png")

## 5. Compression Ratio & Length Correlation

In [ ]:
compression = [s / d for d, s in zip(dial_lens, summ_lens) if d > 0]
print(f"Compression ratio (summary / dialogue tokens):")
print(f"  mean  : {np.mean(compression):.3f}")
print(f"  median: {np.median(compression):.3f}")
print(f"  min   : {np.min(compression):.3f}  max: {np.max(compression):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(compression, bins=50, color="#D65F5F", edgecolor="white", linewidth=0.4)
axes[0].axvline(np.median(compression), color="navy", linestyle="--", linewidth=1.5,
                label=f"median={np.median(compression):.3f}")
axes[0].set_title("Compression Ratio Distribution (train)")
axes[0].set_xlabel("summary_tokens / dialogue_tokens")
axes[0].legend()

axes[1].scatter(dial_lens[:2000], summ_lens[:2000], alpha=0.2, s=8, color="#4878CF")
axes[1].set_title("Dialogue vs Summary Length (first 2000)")
axes[1].set_xlabel("Dialogue tokens")
axes[1].set_ylabel("Summary tokens")

corr = np.corrcoef(dial_lens[:2000], summ_lens[:2000])[0, 1]
axes[1].text(0.05, 0.92, f"Pearson r = {corr:.3f}", transform=axes[1].transAxes, fontsize=10)

plt.tight_layout()
plt.show()

## 6. Load and Inspect data_audit.json

After running `python3 scripts/data_audit.py`, inspect the results here.

In [ ]:
audit_path = PROJECT_ROOT / "results" / "metrics" / "data_audit.json"

if audit_path.exists():
    with open(audit_path) as f:
        audit = json.load(f)

    print("data_audit.json — top-level keys:")
    for k, v in audit.items():
        if not isinstance(v, dict):
            print(f"  {k}: {v}")

    print("\n  Token stats (train):")
    train_ts = audit["token_stats"]["train"]
    df_tok = pd.DataFrame({
        "Field": ["Dialogue tokens", "Summary tokens"],
        **{k: [train_ts["dialogue_tokens"][k], train_ts["summary_tokens"][k]]
           for k in ["min", "p50", "p90", "p95", "p99", "max", "mean"]}
    }).set_index("Field")
    display(df_tok)

    print("\n  Speaker distribution (train):")
    spk = audit["speaker_distribution_train"]
    df_spk2 = pd.DataFrame([{"Speakers": k, **v} for k, v in spk.items()])
    display(df_spk2)

    print(f"\n  Leakage check passed: {audit['leakage_check']['passed']}")
    print(f"  Recommendations: {audit['recommendations']}")
else:
    print("⚠️  data_audit.json not found. Run: python3 scripts/data_audit.py")

## 7. Key Findings Summary

| Finding | Value | Design Implication |
|---|---|---|
| p99 dialogue length | See stats above | `max_source_length=512` covers ≥99% of examples |
| max summary length | See stats above | `max_target_length=128` is safe |
| 2-speaker % | See speaker dist | Model trained on mostly 2-way conversations |
| Compression ratio | ~0.12 | Summaries are ~12% the length of dialogues |
| Data leakage | 0 overlaps | Train/val/test splits are clean |

**⚠️ License Reminder:** SAMSum is CC BY-NC-ND 4.0 — non-commercial use only.  
All training artifacts (weights, generated summaries) inherit this restriction.